# Evaluating Translation Accuracy for Product Localization

Build a translation quality pipeline that evaluates semantic faithfulness, formality register, untranslatable term preservation, and UI string length constraints — using built-in metrics, custom evals, and batch evaluation across language pairs.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/future-agi/cookbooks/blob/main/use-cases/translation-eval.ipynb)

| Time | Difficulty | Features Used |
|------|-----------|---------------|
| 30 min | Intermediate | Evaluation, Custom Eval Metrics, Batch Evaluation |

You're building **LinguaFlow**, a SaaS platform that localizes product UI and documentation into 20+ languages. Your AI translation engine handles button labels, error messages, help text, marketing copy, and legal disclaimers — translating from English into Spanish, French, German, and Japanese.

Machine translation gets the gist right most of the time. But "gist" isn't enough for production UI. A German translation that's 40% longer breaks the button layout. A Spanish translation that uses "tu" instead of "usted" makes your enterprise client's banking app sound like a chatbot. Technical terms like "API" and "OAuth" get transliterated into nonsense. And a legal disclaimer that drops a clause creates actual liability.

This cookbook builds an evaluation pipeline that catches all of these problems before translations go live.

**Prerequisites:**
- FutureAGI account → [app.futureagi.com](https://app.futureagi.com)
- API keys: `FI_API_KEY` and `FI_SECRET_KEY`
- OpenAI API key (`OPENAI_API_KEY`)
- Python 3.9+

In [ ]:
!pip install ai-evaluation futureagi openai

In [ ]:
import os

os.environ["FI_API_KEY"] = "your-fi-api-key"
os.environ["FI_SECRET_KEY"] = "your-fi-secret-key"
os.environ["OPENAI_API_KEY"] = "your-openai-key"

## Step 1: Set up your translation pipeline

A simple OpenAI-based translator with a system prompt that specifies target language, formality level, and domain context. The prompt tells the model to preserve technical terms and keep translations concise for UI strings.

In [ ]:
from openai import OpenAI

client = OpenAI()

def translate(text: str, target_lang: str, string_type: str) -> str:
    """Translate a product string to the target language."""
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0.3,
        messages=[
            {
                "role": "system",
                "content": f"""You are a professional software localization translator.
Translate the following product string from English to {target_lang}.

Rules:
- Use formal register (usted/Sie/vous) for UI strings, error messages, help text, and legal text
- Informal register (tú/du/tu) is acceptable for marketing copy only
- Technical terms (API, SDK, JSON, OAuth, SSO, URL, HTTP) must remain in English
- Keep translations concise — UI strings should not exceed 130% of the source length
- Preserve any placeholder variables like {{name}} or %s exactly as-is
- String type: {string_type}

Return ONLY the translated string, nothing else.""",
            },
            {"role": "user", "content": text},
        ],
    )
    return response.choices[0].message.content.strip()

Translate six English UI strings to Spanish:

In [ ]:
test_strings = [
    {"text": "Save changes", "type": "button_label"},
    {"text": "Error: Your API key is invalid. Please check your credentials and try again.", "type": "error_message"},
    {"text": "Supercharge your workflow with AI-powered analytics", "type": "marketing_headline"},
    {"text": "To configure SSO, navigate to Settings > Security > Single Sign-On. Paste your SAML metadata URL and click Verify. OAuth 2.0 is also supported.", "type": "technical_docs"},
    {"text": "Hover over any chart to see detailed metrics for that time period.", "type": "help_tooltip"},
    {"text": "By proceeding, you agree to our Terms of Service and acknowledge that your data will be processed in accordance with our Privacy Policy.", "type": "legal_disclaimer"},
]

translations = []
for item in test_strings:
    result = translate(item["text"], "Spanish", item["type"])
    translations.append({
        "source": item["text"],
        "translation": result,
        "string_type": item["type"],
    })
    print(f"[{item['type']}]")
    print(f"  EN: {item['text']}")
    print(f"  ES: {result}\n")

## Step 2: Create a test dataset

Six translation pairs with reference translations — the ground truth a human translator would produce. These references let you evaluate both faithfulness to the source and closeness to the ideal translation.

Notice the reference translations follow the same rules: "usted" for formal UI strings, "tu" for marketing, technical terms kept in English. These are the baselines your evals will compare against.

In [ ]:
test_dataset = [
    {
        "source": "Save changes",
        "translation": translations[0]["translation"],
        "reference": "Guardar cambios",
        "string_type": "button_label",
    },
    {
        "source": "Error: Your API key is invalid. Please check your credentials and try again.",
        "translation": translations[1]["translation"],
        "reference": "Error: Su clave de API no es válida. Por favor, verifique sus credenciales e intente de nuevo.",
        "string_type": "error_message",
    },
    {
        "source": "Supercharge your workflow with AI-powered analytics",
        "translation": translations[2]["translation"],
        "reference": "Potencia tu flujo de trabajo con análisis impulsados por IA",
        "string_type": "marketing_headline",
    },
    {
        "source": "To configure SSO, navigate to Settings > Security > Single Sign-On. Paste your SAML metadata URL and click Verify. OAuth 2.0 is also supported.",
        "translation": translations[3]["translation"],
        "reference": "Para configurar SSO, navegue a Configuración > Seguridad > Inicio de sesión único. Pegue su URL de metadatos SAML y haga clic en Verificar. OAuth 2.0 también es compatible.",
        "string_type": "technical_docs",
    },
    {
        "source": "Hover over any chart to see detailed metrics for that time period.",
        "translation": translations[4]["translation"],
        "reference": "Pase el cursor sobre cualquier gráfico para ver las métricas detalladas de ese período.",
        "string_type": "help_tooltip",
    },
    {
        "source": "By proceeding, you agree to our Terms of Service and acknowledge that your data will be processed in accordance with our Privacy Policy.",
        "translation": translations[5]["translation"],
        "reference": "Al continuar, usted acepta nuestros Términos de servicio y reconoce que sus datos serán procesados de acuerdo con nuestra Política de privacidad.",
        "string_type": "legal_disclaimer",
    },
]

## Step 3: Evaluate semantic faithfulness

The core question: does the translation preserve the meaning of the source text? Use `groundedness` (is the translation faithful to the source?) and `completeness` (does it capture all the meaning?). The source text acts as the "context" — the translation should be grounded in and complete relative to it.

A high groundedness score means the translation doesn't add claims that aren't in the source. A high completeness score means it doesn't drop anything. Together, they tell you whether the translation is both accurate and complete.

> **Note:** See [Hallucination Detection with Faithfulness and Groundedness](/docs/cookbook/quickstart/hallucination-detection) for more on catching unsupported claims with `groundedness` and contradictions with `faithfulness`.

In [ ]:
from fi.evals import evaluate

for item in test_dataset:
    results = evaluate(
        ["groundedness", "completeness"],
        output=item["translation"],
        context=item["source"],
        input=item["source"],
        model="turing_small",
    )

    print(f"[{item['string_type']}]")
    print(f"  Source:      {item['source'][:60]}...")
    print(f"  Translation: {item['translation'][:60]}...")
    for r in results:
        status = "PASS" if r.passed else "FAIL"
        print(f"  {r.eval_name:<15} score={r.score}  {status}")
        print(f"    Reason: {r.reason}")
    print()

## Step 4: Create a formality eval

Spanish has two registers: "usted" (formal) and "tu" (informal). Enterprise software should use "usted" for UI strings, but marketing copy can use "tu" to feel more approachable. A built-in metric can't know your formality rules — this is where custom evals come in.

**In the dashboard:**

1. Go to [app.futureagi.com](https://app.futureagi.com) → **Evals** (left sidebar under BUILD)
2. Click **Create Evaluation**
3. Fill in:
   - **Name**: `translation_formality`
   - **Template type**: **Use Future AGI Agents**
   - **Model**: `turing_small`
   - **Output Type**: `Pass/Fail`
4. Write the **Rule Prompt**:

```
You are evaluating a Spanish translation for correct formality register.

Source text (English): {{source_text}}
Translation (Spanish): {{translated_text}}
String type: {{string_type}}

RULES:
- For string types "button_label", "error_message", "help_tooltip", "technical_docs", and "legal_disclaimer": the translation MUST use formal register ("usted", "su", conjugations like "verifique", "haga clic", "navegue").
- For string type "marketing_headline" or "marketing_copy": informal register ("tú", "tu", conjugations like "potencia", "descubre") is acceptable.
- Look for verb conjugations, possessive pronouns, and direct address to determine register.

Mark PASS if the translation uses the correct register for its string type.
Mark FAIL if formal text uses informal register, or if the register cannot be determined because the translation is too short or ambiguous (in which case, note that).
```

5. Click **Create Evaluation**

**Run it via SDK:**

The marketing headline should pass with informal register. Everything else should pass with formal register. If your translator slips into "tu" on an error message, this eval catches it.

> **Note:** See [Custom Eval Metrics: Write Your Own Evaluation Criteria](/docs/cookbook/quickstart/custom-eval-metrics) for the full walkthrough on creating custom evals with Pass/Fail and Percentage output types.

In [ ]:
from fi.evals import Evaluator

evaluator = Evaluator(
    fi_api_key=os.environ["FI_API_KEY"],
    fi_secret_key=os.environ["FI_SECRET_KEY"],
)

for item in test_dataset:
    result = evaluator.evaluate(
        eval_templates="translation_formality",
        inputs={
            "source_text": item["source"],
            "translated_text": item["translation"],
            "string_type": item["string_type"],
        },
    )

    eval_result = result.eval_results[0]
    print(f"[{item['string_type']}]")
    print(f"  Translation: {item['translation'][:70]}...")
    print(f"  Formality: {eval_result.output}")
    print(f"  Reason: {eval_result.reason}\n")

## Step 5: Create an untranslatable terms eval

Technical terms like API, SDK, JSON, OAuth, SSO, SAML, and URL should remain in English. Translating "API" to "interfaz de programacion de aplicaciones" in a button label is wrong. This eval checks that technical terms survive translation intact.

**In the dashboard:**

1. Go to **Evals** → **Create Evaluation**
2. Fill in:
   - **Name**: `untranslatable_terms`
   - **Template type**: **Use Future AGI Agents**
   - **Model**: `turing_small`
   - **Output Type**: `Pass/Fail`
3. Write the **Rule Prompt**:

```
You are evaluating whether a translation correctly preserves technical terms that should NOT be translated.

Source text (English): {{source_text}}
Translation: {{translated_text}}

UNTRANSLATABLE TERMS (must remain exactly as-is in English):
API, SDK, JSON, OAuth, SSO, SAML, URL, HTTP, HTTPS, REST, GraphQL, CLI, CSS, HTML, DNS, IP, TCP, UDP, SMTP, IMAP, FTP, SSH, TLS, SSL, JWT, YAML, XML, SQL, NoSQL, UUID, URI, CDN, GPU, CPU, RAM, SSD, IDE, CI/CD, CORS, WebSocket, Webhook

RULES:
- Every technical term present in the source text must appear unchanged in the translation.
- The terms must appear in their original English form (e.g., "API" not "IPA" or "interfaz de programación").
- Case must be preserved (e.g., "OAuth" not "oauth" or "OAUTH").
- Placeholder variables like {{name}} or %s must also be preserved exactly.

Mark PASS if all technical terms from the source appear unchanged in the translation.
Mark FAIL if any technical term is translated, transliterated, omitted, or has its case changed. List the affected terms.
```

4. Click **Create Evaluation**

**Run it via SDK:**

The technical docs string contains SSO, SAML, URL, and OAuth — all four must appear verbatim in the translation. The error message contains "API" — that must stay too.

In [ ]:
for item in test_dataset:
    result = evaluator.evaluate(
        eval_templates="untranslatable_terms",
        inputs={
            "source_text": item["source"],
            "translated_text": item["translation"],
        },
    )

    eval_result = result.eval_results[0]
    print(f"[{item['string_type']}]")
    print(f"  Source:      {item['source'][:60]}...")
    print(f"  Translation: {item['translation'][:60]}...")
    print(f"  Terms preserved: {eval_result.output}")
    print(f"  Reason: {eval_result.reason}\n")

## Step 6: Create a length constraint eval

UI strings have layout budgets. A German translation is typically 30-40% longer than English. A Spanish translation runs 20-30% longer. If a button label expands beyond what the layout allows, it overflows, wraps, or gets clipped. This eval enforces a 130% length ceiling.

**In the dashboard:**

1. Go to **Evals** → **Create Evaluation**
2. Fill in:
   - **Name**: `translation_length_constraint`
   - **Template type**: **Use Future AGI Agents**
   - **Model**: `turing_small`
   - **Output Type**: `Pass/Fail`
3. Write the **Rule Prompt**:

```
You are evaluating whether a translated string is within acceptable length limits for UI display.

Source text (English): {{source_text}}
Translation: {{translated_text}}
String type: {{string_type}}

LENGTH RULES:
- For "button_label": translation must be within 130% of source character count.
- For "help_tooltip": translation must be within 130% of source character count.
- For "error_message": translation must be within 150% of source character count.
- For "marketing_headline": translation must be within 140% of source character count.
- For "technical_docs" and "legal_disclaimer": no strict length limit (PASS automatically).

Calculate the source length and translation length in characters. Compute the ratio (translation length / source length * 100).

Mark PASS if the ratio is within the allowed percentage for the string type.
Mark FAIL if the ratio exceeds the limit. Report both lengths and the ratio.
```

4. Click **Create Evaluation**

**Run it via SDK:**

Button labels are the most constrained — "Save changes" is 12 characters, and its Spanish equivalent "Guardar cambios" is 16 characters (133%). That's right at the edge. If your translator produces "Guardar los cambios realizados" (30 characters, 250%), this eval flags it.

In [ ]:
for item in test_dataset:
    result = evaluator.evaluate(
        eval_templates="translation_length_constraint",
        inputs={
            "source_text": item["source"],
            "translated_text": item["translation"],
            "string_type": item["string_type"],
        },
    )

    eval_result = result.eval_results[0]
    source_len = len(item["source"])
    trans_len = len(item["translation"])
    ratio = trans_len / source_len * 100

    print(f"[{item['string_type']}]")
    print(f"  Source length:      {source_len} chars")
    print(f"  Translation length: {trans_len} chars ({ratio:.0f}%)")
    print(f"  Within limit: {eval_result.output}")
    print(f"  Reason: {eval_result.reason}\n")

## Step 7: Batch evaluate and identify weak spots

Now run all four evaluations across the full dataset to see which string types and quality dimensions need human review. Upload the dataset, run built-in and custom evals, and download the scored results.

> **Note:** See [Dataset SDK: Upload, Evaluate, and Download Results](/docs/cookbook/quickstart/batch-eval) for the full batch evaluation workflow including programmatic row addition, evaluation statistics, and CSV download.

In [ ]:
import csv
import time
from fi.datasets import Dataset, DatasetConfig
from fi.utils.types import ModelTypes

# Write dataset to CSV for upload
csv_path = "translation_test_data.csv"
with open(csv_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["source", "translation", "reference", "string_type"])
    writer.writeheader()
    for item in test_dataset:
        writer.writerow(item)

# Create the dataset
dataset = Dataset(
    dataset_config=DatasetConfig(
        name="linguaflow-spanish-eval",
        model_type=ModelTypes.GENERATIVE_LLM,
    ),
    fi_api_key=os.environ["FI_API_KEY"],
    fi_secret_key=os.environ["FI_SECRET_KEY"],
)
dataset.create(source=csv_path)
print(f"Dataset created: {dataset.dataset_config.name}")

Run the built-in evaluations:

In [ ]:
dataset.add_evaluation(
    name="groundedness-check",
    eval_template="groundedness",
    required_keys_to_column_names={
        "output": "translation",
        "context": "source",
        "input": "source",
    },
    model="turing_small",
    run=True,
    reason_column=True,
)

dataset.add_evaluation(
    name="completeness-check",
    eval_template="completeness",
    required_keys_to_column_names={
        "output": "translation",
        "input": "source",
    },
    model="turing_small",
    run=True,
    reason_column=True,
)

print("Built-in evaluations started")

Run the custom evaluations:

In [ ]:
dataset.add_evaluation(
    name="formality-check",
    eval_template="translation_formality",
    required_keys_to_column_names={
        "source_text": "source",
        "translated_text": "translation",
        "string_type": "string_type",
    },
    run=True,
    reason_column=True,
)

dataset.add_evaluation(
    name="terms-check",
    eval_template="untranslatable_terms",
    required_keys_to_column_names={
        "source_text": "source",
        "translated_text": "translation",
    },
    run=True,
    reason_column=True,
)

dataset.add_evaluation(
    name="length-check",
    eval_template="translation_length_constraint",
    required_keys_to_column_names={
        "source_text": "source",
        "translated_text": "translation",
        "string_type": "string_type",
    },
    run=True,
    reason_column=True,
)

print("Custom evaluations started")

Download the scored results and analyze:

In [ ]:
df = dataset.download(load_to_pandas=True)

print("Columns:", list(df.columns))
print(df.head())

In [ ]:
# Identify which string types need human review
eval_cols = [c for c in df.columns if "check" in c.lower() and "reason" not in c.lower()]

print("\n=== Translation Quality Summary ===\n")
for _, row in df.iterrows():
    string_type = row["string_type"]
    failures = []
    for col in eval_cols:
        val = row[col]
        if val in [False, "Failed", "failed", "Fail", "fail", 0, 0.0]:
            failures.append(col)

    status = "NEEDS REVIEW" if failures else "AUTO-PUBLISH OK"
    print(f"[{string_type}] {status}")
    if failures:
        for col in failures:
            reason_col = [c for c in df.columns if col.replace("-check", "") in c.lower() and "reason" in c.lower()]
            reason = row[reason_col[0]] if reason_col else "No reason available"
            print(f"  Failed: {col}")
            print(f"  Reason: {reason}")
    print()

The output tells you exactly which translations are safe to auto-publish and which need a human translator's attention. Typical patterns you'll see:

- **Button labels** — often fail length constraints when the translator uses a verbose phrasing
- **Technical docs** — occasionally fail the untranslatable terms check when "SSO" gets expanded to "inicio de sesion unico" instead of staying as "SSO"
- **Marketing copy** — may fail formality if the model uses "usted" when "tu" would be more natural
- **Legal disclaimers** — rarely fail length (no limit) but may fail completeness if a clause gets dropped

To scale this across all your language pairs, repeat the same pipeline for French, German, and Japanese — adjusting the formality rules for each language (vous/tu for French, Sie/du for German, formal/casual conjugation for Japanese).

---

## What you built

You built a translation quality pipeline that evaluates AI translations across four dimensions — semantic faithfulness, formality register, technical term preservation, and UI length constraints — and identifies which translations are safe to auto-publish vs which need human review.

- Translated 6 UI string types from English to Spanish using an OpenAI-based pipeline
- Evaluated semantic quality with `groundedness` and `completeness` (source text as context)
- Created 3 custom evals in the dashboard: formality register, untranslatable terms, and length constraints
- Ran all 5 evaluations as a batch across the full dataset
- Identified which string types and quality dimensions need human review

The same pipeline works for any language pair. Swap the target language, adjust the formality rules (vous/tu for French, Sie/du for German), and re-run. The custom evals are reusable across all your localization projects.